# Set 05 – Steigung m und Achsenabschnitt b interaktiv testen

Training bedeutet: Parameter so verändern, dass der Fehler kleiner wird. In diesem Notebook stellen wir m und b selbst ein. Die Regler zeigen unmittelbar, wie sich Gerade, Residuen und mittlerer quadratischer Fehler verändern.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import FloatSlider, interact
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(12)
x = np.linspace(0, 10, 28)
y = 2.6 * x + 4.0 + rng.normal(0, 2.0, len(x))

## 1. Eine Gerade selbst berechnen

Für gewählte Werte m und b gilt: Vorhersage = m mal x + b. Der mittlere quadratische Fehler ist der Mittelwert der quadrierten Residuen.

In [ ]:
def vorhersage(x_werte, m, b):
    return m * x_werte + b

def mse(y_echt, y_vorhergesagt):
    return np.mean((y_echt - y_vorhergesagt) ** 2)

beispiel = vorhersage(x, m=1.0, b=0.0)
print("MSE für m=1 und b=0:", round(mse(y, beispiel), 2))

## 2. Mehrere Kandidaten vergleichen

Schon wenige Kombinationen zeigen: Sowohl Steigung als auch Achsenabschnitt beeinflussen den Fehler.

In [ ]:
kandidaten = [(1.0, 0.0), (2.0, 4.0), (2.6, 4.0), (3.5, 1.0)]
zeilen = []
for m_wert, b_wert in kandidaten:
    y_hat = vorhersage(x, m_wert, b_wert)
    zeilen.append({"m": m_wert, "b": b_wert, "MSE": mse(y, y_hat)})

display(pd.DataFrame(zeilen).round(2).sort_values("MSE"))

## 3. Regler für m und b

Verschiebe beide Regler und versuche, den MSE möglichst klein zu bekommen. Die dünnen Linien zwischen Punkt und Gerade sind die Residuen.

Falls noch keine Regler angezeigt werden, installiere die Anforderungen erneut und starte den Jupyter-Kernel neu.

In [ ]:
@interact(
    m=FloatSlider(value=1.0, min=-1.0, max=6.0, step=0.1, description="m"),
    b=FloatSlider(value=0.0, min=-10.0, max=15.0, step=0.5, description="b"),
)
def zeichne_gerade(m, b):
    y_hat = vorhersage(x, m, b)
    fehler = mse(y, y_hat)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(x, y, color="#4C78A8", label="Daten")
    ax.plot(x, y_hat, color="#E45756", linewidth=2.5, label=f"y = {m:.1f}x + {b:.1f}")
    ax.vlines(x, y, y_hat, color="gray", alpha=0.35)
    ax.set_ylim(y.min() - 5, y.max() + 5)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Mittlerer quadratischer Fehler: {fehler:.2f}")
    ax.legend()
    plt.show()

## 4. Optimum von scikit-learn

Nun lassen wir LinearRegression die Parameter automatisch aus den Daten bestimmen und vergleichen sie mit der eigenen Einstellung.

In [ ]:
X = pd.DataFrame({"x": x})
modell = LinearRegression()
modell.fit(X, y)

m_optimal = modell.coef_[0]
b_optimal = modell.intercept_
mse_optimal = mse(y, modell.predict(X))

print(f"scikit-learn: m={m_optimal:.3f}, b={b_optimal:.3f}, MSE={mse_optimal:.3f}")

## 5. Fehlerlandschaft

Jede Kombination aus m und b besitzt einen Fehlerwert. Die Farbkarte zeigt diese Fehlerlandschaft; der weiße Punkt markiert die von scikit-learn gefundene Lösung.

In [ ]:
m_werte = np.linspace(0.5, 4.5, 100)
b_werte = np.linspace(-3, 11, 100)
landschaft = np.empty((len(b_werte), len(m_werte)))

for i, b_wert in enumerate(b_werte):
    for j, m_wert in enumerate(m_werte):
        landschaft[i, j] = mse(y, vorhersage(x, m_wert, b_wert))

fig, ax = plt.subplots(figsize=(9, 5))
bild = ax.contourf(m_werte, b_werte, landschaft, levels=30, cmap="viridis")
ax.scatter(m_optimal, b_optimal, color="white", edgecolor="black", s=90, label="Minimum")
ax.set_xlabel("Steigung m")
ax.set_ylabel("Achsenabschnitt b")
ax.set_title("MSE-Fehlerlandschaft")
ax.legend()
fig.colorbar(bild, ax=ax, label="MSE")
plt.show()

Das interaktive Verschieben simuliert die Grundidee der Optimierung. In der Praxis durchsucht LinearRegression nicht einfach alle Reglerpositionen, sondern bestimmt die Least-Squares-Lösung rechnerisch.